# Umbral de confianza, recall por imagen y falsas alarmas (YOLO26, iteración 1)

Calcula las métricas que Ultralytics no reporta y que `criterios_exito_provisional.md` (apartado 1.1) pone como principales:

- **Umbral de operación**: el más alto que todavía conserva un recall de cajas de 0.90 en validación.
- **Recall por imagen**: de las fotos con oso, en cuántas el modelo pone al menos una detección por encima del umbral. Meta: 0.95 o más.
- **Falsas alarmas**: de las fotos sin oso, cuántas reciben alguna detección por encima del umbral. Meta: menos del 10 %, por fuente.
- Lo mismo separando día y noche en ENA24.

Usa solo el conjunto de validación, salvo que se active `EVALUAR_PRUEBA` en la celda 0.

Configuración del cuaderno: GPU T4, **Internet activado**, y los datasets `deteccionoso` (imágenes) y `yolodeteccionoso` (pesos) adjuntos.

## 0. Parámetros

In [ ]:
UMBRAL_SOBRE    = "ena24"   # partición de validación donde se exige el recall de 0.90: "ena24" o "global"
RECALL_OBJETIVO = 0.90      # criterios de éxito, apartado 1.1
IOU_MIN         = 0.5       # una detección acierta si su caja se solapa al menos 0.5 (IoU) con una caja real
CONF_MIN        = 0.001     # se guardan todas las detecciones desde esta confianza, igual que la validación de Ultralytics
EVALUAR_PRUEBA  = False     # dejar en False: el conjunto de prueba se usa una sola vez, con los modelos finales

assert UMBRAL_SOBRE in ("ena24", "global")
print(f"Umbral elegido sobre validación '{UMBRAL_SOBRE}' con recall de cajas >= {RECALL_OBJETIVO}, IoU >= {IOU_MIN}")
print("Conjuntos a evaluar:", ["val", "test"] if EVALUAR_PRUEBA else ["val"])

## 1. Entorno

In [ ]:
%pip install -q ultralytics==8.4.138

import sys, torch, ultralytics, numpy, PIL
print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Ultralytics :", ultralytics.__version__)
print("numpy       :", numpy.__version__, "| Pillow:", PIL.__version__)

## 2. Rutas y modelos

Busca todos los archivos `*_best.pt` dentro de los datasets adjuntos y deduce de cada nombre el tamaño, la rotación y la semilla.

In [ ]:
import glob, re
from pathlib import Path

DATASET = Path("/kaggle/input/datasets/josemartinvallejos/deteccionoso")
assert DATASET.is_dir(), f"No existe {DATASET}. Revisar que el dataset de imágenes esté adjunto."

rutas_pt = sorted(Path(p) for p in glob.glob("/kaggle/input/**/*_best.pt", recursive=True))
if not rutas_pt:
    print("No se encontró ningún *_best.pt. Contenido de /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*/*/*")):
        print("  ", p)
    raise SystemExit("Adjuntar el dataset con los pesos y volver a ejecutar.")

TAMANO = {"n": "nano", "s": "small", "m": "medium"}
ORDEN = {"nano": 0, "small": 1, "medium": 2}
MODELOS = []
for p in rutas_pt:
    run_id = p.name[: -len("_best.pt")]
    mt = re.match(r"^(e\d+b?)_([nsm])_deg(\d+)_s(\d+)$", run_id)
    if not mt:
        print("AVISO: nombre no reconocido, se omite:", p)
        continue
    MODELOS.append({"run_id": run_id, "tamano": TAMANO[mt.group(2)], "degrees": int(mt.group(3)),
                    "semilla": int(mt.group(4)), "ruta": p})
MODELOS.sort(key=lambda d: (ORDEN[d["tamano"]], d["degrees"], d["semilla"]))

print(f"Imágenes: {DATASET}")
print(f"{len(MODELOS)} modelos:")
for d in MODELOS:
    print(f"   {d['run_id']:<16} {d['tamano']:<7} degrees={d['degrees']:<3} semilla={d['semilla']}  {d['ruta']}")

## 3. Imágenes y etiquetas

Para cada imagen: fuente (por el prefijo del nombre), cajas reales (desde `labels/`), si tiene oso, y si es nocturna. Nocturna se decide con el mismo criterio de `describir_dataset.py`: imagen reducida a 64×64 cuya diferencia media entre canales de color es menor que 2 (es decir, casi sin color, típico del infrarrojo).

Los conteos se comparan con los de `informe_preparacion_datos.md`. Si alguno no coincide, el cuaderno lo avisa. Las nocturnas se comparan solo en ENA24, porque `dividir_dataset.py` marcó todas las de iNaturalist como diurnas; las fotos de iNaturalist en blanco y negro se informan aparte y no se usan en las particiones de día y noche.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image, ImageChops, ImageStat

ESPERADO = {
    "val":  {"imagenes": 598, "positivas": 326, "cajas": 351, "ena24": 338, "inat": 260, "nocturnas": 224,
             "ena24_pos_dia": 52, "ena24_pos_noche": 104, "ena24_neg_dia": 62, "ena24_neg_noche": 120,
             "inat_pos": 170, "inat_neg": 90},
    "test": {"imagenes": 578, "positivas": 313, "cajas": 349, "ena24": 318, "inat": 260, "nocturnas": 197,
             "ena24_pos_dia": 59, "ena24_pos_noche": 86, "ena24_neg_dia": 62, "ena24_neg_noche": 111,
             "inat_pos": 168, "inat_neg": 92},
}

def es_nocturna(ruta):
    """Mismo criterio que describir_dataset.py (escala_grises)."""
    with Image.open(ruta) as img:
        try:
            img.draft("RGB", (400, 400))
        except Exception:
            pass
        peq = img.convert("RGB").resize((64, 64))
    r, g, b = peq.split()
    dif = (ImageStat.Stat(ImageChops.difference(r, g)).mean[0]
           + ImageStat.Stat(ImageChops.difference(g, b)).mean[0]) / 2
    return round(dif, 2) < 2.0

def fuente_de(nombre):
    if nombre.startswith("ena24_"):
        return "ena24"
    if nombre.startswith(("inat_", "inatneg_")):
        return "inat"
    return "otra"

def leer_cajas(ruta_txt):
    """Cajas YOLO (cx, cy, w, h normalizadas) convertidas a esquinas (x1, y1, x2, y2)."""
    if not ruta_txt.exists():
        return np.zeros((0, 4))
    filas = [l.split() for l in ruta_txt.read_text().splitlines() if l.strip()]
    if not filas:
        return np.zeros((0, 4))
    a = np.array([[float(v) for v in f[1:5]] for f in filas])
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def tabla_imagenes(conjunto):
    filas, cajas = [], {}
    for ruta in sorted((DATASET / "images" / conjunto).glob("*.jpg")):
        gt = leer_cajas(DATASET / "labels" / conjunto / (ruta.stem + ".txt"))
        cajas[ruta.name] = gt
        filas.append({"conjunto": conjunto, "archivo": ruta.name, "ruta": str(ruta),
                      "fuente": fuente_de(ruta.name), "n_cajas": len(gt), "positiva": len(gt) > 0,
                      "nocturna": es_nocturna(ruta)})
    return pd.DataFrame(filas), cajas

def comprobar(conjunto, df):
    e = df[df.fuente == "ena24"]
    i = df[df.fuente == "inat"]
    obtenido = {
        "imagenes": len(df), "positivas": int(df.positiva.sum()), "cajas": int(df.n_cajas.sum()),
        "ena24": len(e), "inat": len(i), "nocturnas": int(e.nocturna.sum()),
        "ena24_pos_dia": int((e.positiva & ~e.nocturna).sum()), "ena24_pos_noche": int((e.positiva & e.nocturna).sum()),
        "ena24_neg_dia": int((~e.positiva & ~e.nocturna).sum()), "ena24_neg_noche": int((~e.positiva & e.nocturna).sum()),
        "inat_pos": int(i.positiva.sum()), "inat_neg": int((~i.positiva).sum()),
    }
    ok = True
    for k, v in obtenido.items():
        esp = ESPERADO[conjunto][k]
        marca = "" if v == esp else f"   <-- ATENCIÓN, se esperaba {esp}"
        ok &= v == esp
        print(f"   {k:<16} {v:>5}{marca}")
    # las listas por fuente deben contener exactamente las imágenes de cada fuente
    for fuente in ("ena24", "inat"):
        lista = DATASET / f"{conjunto}_{fuente}.txt"
        if lista.exists():
            en_lista = {Path(l.strip()).name for l in lista.read_text(encoding="utf-8").splitlines() if l.strip()}
            iguales = en_lista == set(df[df.fuente == fuente].archivo)
            ok &= iguales
            print(f"   lista {lista.name:<15} {'coincide' if iguales else 'NO COINCIDE con el prefijo del nombre'}")
    print(f"   (informativo) fotos de iNaturalist en blanco y negro: {int(i.nocturna.sum())}; "
          "no entran en ninguna partición de día o noche")
    print("   Todo coincide." if ok else "   Hay diferencias: revisar antes de interpretar los resultados.")

CONJUNTOS = ["val", "test"] if EVALUAR_PRUEBA else ["val"]
IMGS, CAJAS = {}, {}
for conj in CONJUNTOS:
    IMGS[conj], CAJAS[conj] = tabla_imagenes(conj)
    print(f"Conjunto '{conj}':")
    comprobar(conj, IMGS[conj])

## 4. Detecciones de cada modelo

Cada modelo procesa las imágenes una por una (lote de 1), con confianza mínima `CONF_MIN`. Después cada detección se marca como acierto o no: se recorren de mayor a menor confianza y cada una se asigna a la caja real libre con la que más se solapa, si ese solapamiento es de al menos `IOU_MIN`. Recorrerlas por confianza es lo que hace que el conteo sea exacto para cualquier umbral: lo que pasa por encima de un umbral no depende de lo que hay por debajo.

In [ ]:
import time
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

LISTAS = Path("/kaggle/working/listas")
LISTAS.mkdir(parents=True, exist_ok=True)

def predecir(modelo, df, conj):
    # Se pasa un archivo de texto con las rutas, no una lista de Python. Con una lista, Ultralytics
    # abre todas las imágenes a la vez y las procesa en un solo lote, lo que agota la memoria de la GPU.
    # Con el archivo de texto lee una imagen por vez (lote de 1), igual que la aplicación final.
    lista = LISTAS / f"{conj}.txt"
    lista.write_text("\n".join(df["ruta"]) + "\n", encoding="utf-8")
    trozos, vistas = [], 0
    for r in modelo.predict(source=str(lista), batch=1, conf=CONF_MIN, imgsz=640, device=DEVICE,
                            stream=True, verbose=False):
        vistas += 1
        n = len(r.boxes)
        if n == 0:
            continue
        conf = r.boxes.conf.cpu().numpy()
        xy = r.boxes.xyxyn.cpu().numpy()
        trozos.append(pd.DataFrame({"archivo": [Path(r.path).name] * n, "conf": conf,
                                    "x1": xy[:, 0], "y1": xy[:, 1], "x2": xy[:, 2], "y2": xy[:, 3]}))
    assert vistas == len(df), f"Se procesaron {vistas} imágenes de {len(df)}"
    if not trozos:
        return pd.DataFrame(columns=["archivo", "conf", "x1", "y1", "x2", "y2"])
    return pd.concat(trozos, ignore_index=True)

def iou_matriz(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)

def emparejar(pred, cajas):
    tp = np.zeros(len(pred), dtype=bool)
    for nombre, idx in pred.groupby("archivo").indices.items():
        gt = cajas.get(nombre)
        if gt is None or len(gt) == 0:
            continue
        orden = idx[np.argsort(-pred["conf"].values[idx], kind="stable")]
        ious = iou_matriz(pred[["x1", "y1", "x2", "y2"]].values[orden], gt)
        libre = np.ones(len(gt), dtype=bool)
        for k, fila in enumerate(ious):
            fila = np.where(libre, fila, -1.0)
            j = int(fila.argmax())
            if fila[j] >= IOU_MIN:
                libre[j] = False
                tp[orden[k]] = True
    return tp

PRED = {}
for d in MODELOS:
    modelo = YOLO(str(d["ruta"]))
    for conj in CONJUNTOS:
        t0 = time.time()
        p = predecir(modelo, IMGS[conj], conj)
        p["tp"] = emparejar(p, CAJAS[conj])
        PRED[(d["run_id"], conj)] = p
        print(f"{d['run_id']:<16} {conj:<5} {len(p):>7} detecciones, {int(p.tp.sum()):>4} aciertos "
              f"({time.time() - t0:.0f} s)")
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Umbral y métricas

Para cada modelo:
1. Umbral: la confianza más alta con la que el recall de cajas en validación (partición `UMBRAL_SOBRE`) sigue siendo al menos `RECALL_OBJETIVO`.
2. Con ese umbral, en cada partición: precisión, recall y F1 de cajas; recall por imagen; falsas alarmas.

Los porcentajes llevan un intervalo de confianza del 95 % (método de Wilson), porque con 100 a 300 imágenes por partición el margen es de varios puntos, como advierten los criterios (apartado 1.0).

Control de calidad: con las mismas detecciones se calcula mAP50 usando la función de Ultralytics (`compute_ap`) y se compara con el que reportó la validación de cada modelo. Deberían diferir poco: la validación de Ultralytics prepara las imágenes en lotes y empareja las cajas de otra forma. Medido en validación el 30 de septiembre de 2026 con los seis modelos: hasta 0.007 en global y en ENA24, con signo variable, y hasta 0.015 en iNaturalist, siempre por debajo del valor de Ultralytics. Causa probable, no verificada: la validación de Ultralytics procesa las fotos en lotes de 16 agrupados por forma y les añade un borde extra, mientras que este cuaderno procesa cada foto sola, como lo hará la aplicación; en iNaturalist, con fotos horizontales y verticales mezcladas, ese borde cambia más. Diferencias mucho mayores que estas sí indicarían un problema.

In [ ]:
import math
from ultralytics.utils.metrics import compute_ap

REF_MAP50 = {  # mAP50 de la celda 5 del cuaderno de entrenamiento (validación de Ultralytics)
    "e1_n_deg0_s0":  {"global": 0.9318, "ena24": 0.9550, "inat": 0.9076},
    "e1b_n_deg0_s1": {"global": 0.9285, "ena24": 0.9325, "inat": 0.9220},
    "e3_s_deg0_s0":  {"global": 0.9352, "ena24": 0.9457, "inat": 0.9198},
    "e3b_s_deg0_s1": {"global": 0.9409, "ena24": 0.9574, "inat": 0.9302},
    "e4_m_deg0_s0":  {"global": 0.9330, "ena24": 0.9573, "inat": 0.9118},
    "e4b_m_deg0_s1": {"global": 0.9363, "ena24": 0.9585, "inat": 0.9165},
}
PARTICIONES = ["global", "ena24", "inat", "ena24_dia", "ena24_noche"]

def mascara(df, particion):
    if particion == "global":
        return np.ones(len(df), dtype=bool)
    if particion in ("ena24", "inat"):
        return (df.fuente == particion).values
    if particion == "ena24_dia":
        return ((df.fuente == "ena24") & ~df.nocturna).values
    if particion == "ena24_noche":
        return ((df.fuente == "ena24") & df.nocturna).values
    raise ValueError(particion)

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / den
    margen = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centro - margen), min(1.0, centro + margen))

def ap50(p, n_gt):
    if n_gt == 0 or len(p) == 0:
        return float("nan")
    orden = np.argsort(-p["conf"].values, kind="stable")
    tp = p["tp"].values[orden]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    return float(compute_ap(ctp / n_gt, ctp / np.maximum(ctp + cfp, 1))[0])

def umbral_para_recall(p, n_gt, objetivo):
    k = math.ceil(round(objetivo * n_gt, 9))
    conf_tp = np.sort(p.loc[p.tp, "conf"].values)[::-1]
    if n_gt == 0 or len(conf_tp) < k:
        return float("nan")
    return float(conf_tp[k - 1])

def metricas(run_id, conj, particion, t):
    df = IMGS[conj]
    m = mascara(df, particion)
    imgs = df[m]
    p = PRED[(run_id, conj)]
    p = p[p.archivo.isin(set(imgs.archivo))]
    n_gt = int(imgs.n_cajas.sum())
    sel = p[p.conf >= t]
    n_pred, n_tp = len(sel), int(sel.tp.sum())
    P = n_tp / n_pred if n_pred else float("nan")
    R = n_tp / n_gt if n_gt else float("nan")
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else float("nan")
    max_conf = imgs.archivo.map(p.groupby("archivo").conf.max()).fillna(0.0).values
    alarma = max_conf >= t
    pos = imgs.positiva.values
    k_pos, n_pos = int((alarma & pos).sum()), int(pos.sum())
    k_neg, n_neg = int((alarma & ~pos).sum()), int((~pos).sum())
    ri_lo, ri_hi = wilson(k_pos, n_pos)
    fa_lo, fa_hi = wilson(k_neg, n_neg)
    return {"conjunto": conj, "particion": particion, "umbral": t,
            "n_cajas": n_gt, "n_detecciones": n_pred, "aciertos": n_tp, "P_caja": P, "R_caja": R, "F1_caja": F1,
            "n_pos": n_pos, "pos_detectadas": k_pos, "recall_imagen": k_pos / n_pos if n_pos else float("nan"),
            "ri_ic_inf": ri_lo, "ri_ic_sup": ri_hi,
            "n_neg": n_neg, "neg_con_alarma": k_neg, "falsas_alarmas": k_neg / n_neg if n_neg else float("nan"),
            "fa_ic_inf": fa_lo, "fa_ic_sup": fa_hi,
            "ap50": ap50(p, n_gt)}

UMBRAL, UMBRAL_ALT, filas = {}, {}, []
otra = "global" if UMBRAL_SOBRE == "ena24" else "ena24"
for d in MODELOS:
    rid = d["run_id"]
    pv, dv = PRED[(rid, "val")], IMGS["val"]
    def n_y_p(part):
        imgs = dv[mascara(dv, part)]
        return pv[pv.archivo.isin(set(imgs.archivo))], int(imgs.n_cajas.sum())
    UMBRAL[rid] = umbral_para_recall(*n_y_p(UMBRAL_SOBRE), RECALL_OBJETIVO)
    UMBRAL_ALT[rid] = umbral_para_recall(*n_y_p(otra), RECALL_OBJETIVO)
    for conj in CONJUNTOS:
        for part in PARTICIONES:
            f = metricas(rid, conj, part, UMBRAL[rid])
            f.update({"run_id": rid, "tamano": d["tamano"], "semilla": d["semilla"], "umbral_sobre": UMBRAL_SOBRE})
            filas.append(f)
RES = pd.DataFrame(filas)

print(f"Umbral elegido sobre validación '{UMBRAL_SOBRE}' (y, solo como referencia, el que saldría sobre '{otra}'):")
for d in MODELOS:
    print(f"   {d['run_id']:<16} {UMBRAL[d['run_id']]:.3f}   (sobre {otra}: {UMBRAL_ALT[d['run_id']]:.3f})")

print("\nControl de calidad, mAP50 en validación: este cuaderno frente a Ultralytics")
for d in MODELOS:
    rid = d["run_id"]
    for part in ("global", "ena24", "inat"):
        mio = RES[(RES.run_id == rid) & (RES.conjunto == "val") & (RES.particion == part)].ap50.iloc[0]
        ref = REF_MAP50.get(rid, {}).get(part)
        dif = f"{mio - ref:+.4f}" if ref is not None else "sin referencia"
        print(f"   {rid:<16} {part:<7} {mio:.4f}  Ultralytics {ref if ref is not None else '-'}  diferencia {dif}")

## 6. Tablas de resultados

Las fracciones se muestran como aciertos/total, proporción y [intervalo del 95 %].

In [ ]:
def frac(k, n, lo, hi):
    return f"{k}/{n} = {k / n:.2f} [{lo:.2f}-{hi:.2f}]" if n else "-"

def tabla(conj, particion):
    sub = RES[(RES.conjunto == conj) & (RES.particion == particion)]
    return pd.DataFrame({
        "modelo": sub.run_id, "umbral": sub.umbral.round(3),
        "P caja": sub.P_caja.round(3), "R caja": sub.R_caja.round(3), "F1": sub.F1_caja.round(3),
        "recall por imagen": [frac(r.pos_detectadas, r.n_pos, r.ri_ic_inf, r.ri_ic_sup) for r in sub.itertuples()],
        "falsas alarmas": [frac(r.neg_con_alarma, r.n_neg, r.fa_ic_inf, r.fa_ic_sup) for r in sub.itertuples()],
    }).to_string(index=False)

TITULOS = {"ena24": "ENA24 (resultado principal)", "inat": "iNaturalist (optimista)", "global": "Global",
           "ena24_dia": "ENA24 de día", "ena24_noche": "ENA24 de noche"}
for conj in CONJUNTOS:
    for part in PARTICIONES:
        print(f"\n=== {conj} | {TITULOS[part]} ===")
        print(tabla(conj, part))

print("\nMetas (criterios, apartado 1.1): recall por imagen >= 0.95; falsas alarmas < 0.10 por fuente; "
      "precisión de cajas >= 0.70.")

## 7. Archivos para descargar

- `umbral_metricas.csv`: todas las cifras de las tablas, una fila por modelo, conjunto y partición.
- `umbral_imagenes.csv`: una fila por imagen con su fuente, si tiene oso, si es nocturna y la confianza máxima de cada modelo. Con este archivo se puede recalcular el recall por imagen y las falsas alarmas para cualquier umbral, sin GPU.
- `umbral_detecciones.csv.gz`: las detecciones con confianza de 0.01 o más, marcadas como acierto o no. Sirve para rehacer las curvas en local.

In [ ]:
OUT = Path("/kaggle/working")
RES.to_csv(OUT / "umbral_metricas.csv", index=False)

imgs_out = []
for conj in CONJUNTOS:
    df = IMGS[conj].drop(columns=["ruta"]).copy()
    for d in MODELOS:
        p = PRED[(d["run_id"], conj)]
        df["maxconf_" + d["run_id"]] = df.archivo.map(p.groupby("archivo").conf.max()).fillna(0.0).round(4)
    imgs_out.append(df)
pd.concat(imgs_out, ignore_index=True).to_csv(OUT / "umbral_imagenes.csv", index=False)

det_out = []
for (rid, conj), p in PRED.items():
    q = p[p.conf >= 0.01].copy()
    q.insert(0, "conjunto", conj)
    q.insert(0, "run_id", rid)
    det_out.append(q)
pd.concat(det_out, ignore_index=True).round(4).to_csv(OUT / "umbral_detecciones.csv.gz", index=False)

for nombre in ("umbral_metricas.csv", "umbral_imagenes.csv", "umbral_detecciones.csv.gz"):
    print(f"   {nombre:<28} {(OUT / nombre).stat().st_size / 1e6:.2f} MB")